# Wheat Scene

In [26]:
import os
import random

import pandas as pd
from matplotlib import pyplot as plt
from ipywidgets import interactive, FloatText
from openalea.archicrop.archicrop import ArchiCrop
from openalea.archicrop.display import build_scene
from openalea.archicrop.stand import agronomic_plot
from openalea.plantgl.all import Color3, Material
from openalea.archicrop.viability import leaf_area_plant
from openalea.mtg.io import write_mtg
from pathlib import Path
%gui qt

In [16]:
def fix_obj(fn):
    """Fix the OBJ file by removing 'vn ' and rewriting 'f ' without 'vn '."""
    with open(fn, "r") as f:
        lines = f.readlines()

    out = []

    for line in lines:
        if line.startswith("vn "):
            continue
        if line.startswith("f "):
            tokens = line.split()
            new_face = ["f"]

            for t in tokens[1:]:
                parts = t.split("/")
                if len(parts) == 3:
                    # v/vt/vn -> v/vt
                    if parts[1]:
                        new_face.append("/".join(parts[:2]))
                    else:
                        new_face.append(parts[0])
                elif len(parts) == 2:
                    new_face.append(t)
                else:
                    new_face.append(parts[0])

            out.append(" ".join(new_face) + "\n")
        else:
            sline = line.strip()
            if sline:
                out.append(line)

    with open(fn, "w") as f:
        f.writelines(out)


def save_mtg(g, scene, mtg_fn, obj_fn):
    """Saves MTG and PlantGL scene into MTG and OBJ files."""
    g.properties()['Id']={}
    ids = g.properties()['Id']
    for vid in g.vertices(): 
        ids[vid]=vid

    def bools(g):
        for p in ['is_green', 'grow', 'dead', 'senescence']:
            _prop = g.properties()[p]
            for vid in _prop:
                _prop[vid] = int(bool(_prop[vid]))
    
    bools(g)
    props = [
        ('Id','INT'),
     ('rank', 'INT'),
     ('length','REAL'),
     ('visible_length','REAL'),
     ('is_green','INT'),
     ('stem_diameter','REAL'),
     ('azimuth','REAL'),
     ('grow','INT'),
     ('age','REAL'),
     ('tiller_angle','REAL'),
     ('leaf_area','REAL'),
     ('visible_leaf_area','REAL'),
     ('senescent_area','REAL'),
     ('senescent_length','REAL'),
     ('shape_max_width','REAL'),
     ('dead','INT'),
     ('inclination','REAL'),
     ('senescence', 'INT')
    ] 

    mtg_lines=write_mtg(g, props)
    fn = mtg_fn
    if fn.exists():
        fn.unlink(missing_ok=True)

    fn.write_text(mtg_lines)
    print(f'Write MTG in {str(fn)}')

    for sh in scene:
        sh.setName(f'vid_{sh.id}')

    fn = obj_fn
    if fn.exists():
        fn.unlink(missing_ok=True)
    scene.save(str(obj_fn))

    fix_obj(obj_fn)

In [17]:
# Set ArchiCrop parameters
archi = {
    "nb_phy": 10, # number of phytomers on the main stem : [10,40] (Ndiaye et al., 2021; Lafarge and Tardieu, 2002; Clerget, 2008; Ganeme et al., 2022)
    "nb_short_phy": 5,
    "short_phy_len": 3,

    # Stem
    "height": 90,
	"stem_q": 1, # parameter for ligule height distribution along axis : [1.1] (Kaitaniemi et al., 1999)
    "diam_base": 0.8, # stem base diameter : [2.2] (Ndiaye et al., 2021)
    "diam_top": 0.3, # stem top diameter: [1.2] (Ndiaye et al., 2021)

    # Leaf area distribution along the stem
    "leaf_area": 1500,
    "rmax": 0.83, # parameter for leaf area distribution along axis : [0.6,0.8] (Kaitaniemi et al., 1999; Welcker et al.)
    "skew": 0.001, # parameter for leaf area distribution along axis : [0.0005,0.1] (Kaitaniemi et al., 1999; Welcker et al.)

    # blade area
    "wl": 0.079, # leaf blade width-to-length ratio :
    "klig": 0.6, # parameter for leaf blade shape
    "swmax": 0.55, # parameter for leaf blade shape
    "f1": 0.64, # parameter for leaf blade shape
    "f2": 0.92, # parameter for leaf blade shape

    # Leaf blade position in space
    "insertion_angle": 73, # leaf blade insertion angle : [10,50] (Truong et al., 2015; Kaitaniemi et al., 1999)
    "scurv": 0.32, # leaf blade relative inflexion point : [0.6, 0.8] ()
    "curvature": 56, # leaf blade insertion-to-tip angle : [0,130] [45, 135] (Kaitaniemi et al., 1999)
    "phyllotactic_angle": 180, # phyllotactic angle : [180] (Davis et al., 2024)
    "phyllotactic_deviation": 90, # half-deviation to phyllotactic angle : [0,90] (Davis et al., 2024)

    # Development
    "phyllochron": 40, # [30,70], # phyllochron, i.e. stem element appearance rate : [40,65 then x1.6-2.5] (Clerget, 2008)
    # "plastochron": [40,65], # plastochron, i.e. leaf blade appearance rate : [34,46 then 80-93] (Rami Kumar et al., 2009)
    "stem_duration": 1.6,
    "leaf_duration": 1.6,

    # Tillering
    "nb_tillers": 4, # number of tillers : [0,6] (Lafarge et al., 2002)
    "tiller_angle": 10,
    "tiller_delay": 1, # delay, as factor of phyllochron, between the appearance of a phytomer and the appearance of its tiller : [] ()
    "reduction_factor": 0.8, # reduction factor between tillers of consecutive order : [0.8,1] ()
    "tropism_coefficient": 0.12
    }
archi

{'nb_phy': 10,
 'nb_short_phy': 5,
 'short_phy_len': 3,
 'height': 90,
 'stem_q': 1,
 'diam_base': 0.8,
 'diam_top': 0.3,
 'leaf_area': 1500,
 'rmax': 0.83,
 'skew': 0.001,
 'wl': 0.079,
 'klig': 0.6,
 'swmax': 0.55,
 'f1': 0.64,
 'f2': 0.92,
 'insertion_angle': 73,
 'scurv': 0.32,
 'curvature': 56,
 'phyllotactic_angle': 180,
 'phyllotactic_deviation': 90,
 'phyllochron': 40,
 'stem_duration': 1.6,
 'leaf_duration': 1.6,
 'nb_tillers': 4,
 'tiller_angle': 10,
 'tiller_delay': 1,
 'reduction_factor': 0.8,
 'tropism_coefficient': 0.12}

In [18]:
plant = ArchiCrop(**archi)
plant.generate_potential_plant()
g = plant.g

In [19]:
def make_scene(length, width, density, inter_row, noise):
	m = Material(Color3(0,80,0))
	print(f"leaf area plant:{leaf_area_plant(g)}")

	nplants, positions, domain, domain_area, unit = agronomic_plot(length=length, width=width, density=density, inter_row=inter_row, noise=noise)
	print(f"nb plants: {nplants}")
	scene, labels = build_scene(g, positions, orientation=[360*random.gauss() for i in range(len(positions))], leaf_material=m, stem_material=m, senescence=True)

	print(f"leaf area canopy:{leaf_area_plant(g) * nplants}")
	return scene

In [ ]:
w = interactive(make_scene, {'manual': True}, length=FloatText(description='Length:', value=4),
 width=FloatText(description='Width:', value=2), density=FloatText(description='Density:', value=70),
 inter_row=FloatText(description='Inter row:', value=0.175), 
 noise=FloatText(description='Noise:', value=0.01))
w

interactive(children=(FloatText(value=4.0, description='Length:'), FloatText(value=2.0, description='Width:'),…

In [23]:
from openalea.plantgl.all import Viewer
Viewer.display(w.result)

In [24]:
w.result.save('wheat_scene.obj')

Write wheat_scene.obj
Write wheat_scene.mtl


In [29]:
fix_obj(Path('wheat_scene.obj'))
save_mtg(g, w.result, Path('wheat_scene.mtg'), Path('wheat_scene.obj'))

Write MTG in wheat_scene.mtg
Write wheat_scene.obj
Write wheat_scene.mtl
